# 01 — Download and prepare NHANES testosterone data

**Why NHANES and not MHANES.** Hennigar et al. (JAMA, 15 Oct 2026,
doi:10.1001/jama.2026.19794) analysed the Military Health and Nutrition
Examination Study (MHANES), a Pennington Biomedical / USARIEM survey of
648 active-duty Army soldiers at five installations (Aug 2024 – Feb 2026).
There is **no public-use file**: ClinicalTrials.gov NCT06380322 lists
*IPD sharing: No*, and the protocol paper (Berryman et al., Curr Dev Nutr
2026) says deidentified data "may be made available to external
investigators through a controlled-access process" under DoD data
governance.

The closest public substitute is the civilian NHANES, which MHANES was
explicitly modelled on. NHANES 2021–2023 measured total testosterone and
SHBG by LC-MS/MS on everyone aged 3+ who gave blood, records whether the
exam was in the **morning (fasting) session**, and carries body-measures
and design variables. This notebook pulls those files (plus the
2011–2016 cycles for a pooled sensitivity check) and writes one tidy
parquet of men aged 18–59 with a testosterone result.

Files (all from `wwwn.cdc.gov/Nchs/Data/Nhanes/Public/<year>/DataFiles/`):

| file | contents used |
|---|---|
| `TST_*` | `LBXTST` total testosterone (ng/dL), `LBXSHBG` SHBG (nmol/L), `WTPH2YR` phlebotomy weight (2021–23 only) |
| `DEMO_*` | age, sex, race/ethnicity (`RIDRETH3`), `SDMVSTRA`/`SDMVPSU`, `WTMEC2YR` |
| `BMX_*` | `BMXBMI` |
| `FASTQX_*` | `PHDSESN`/`PHDSESNZ` exam session (0 = morning), `PHAFSTHR` fasting hours |
| `BIOPRO_*` | `LBXSAL` albumin (g/dL) for calculated free testosterone |
| `RXQ_RX_*` | `RXDDRUG` prescription names, to flag testosterone therapy (names not released for 2021–23) |


In [1]:
import os, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
from survey_utils import vermeulen_free_t

DATA_DIR = Path(os.path.abspath(os.path.join('..', 'data')))
RAW_DIR = DATA_DIR / 'raw' / 'nhanes'
DERIVED_DIR = DATA_DIR / 'derived'
DERIVED_DIR.mkdir(parents=True, exist_ok=True)

CYCLES = {  # cycle label: (URL year folder, file suffix)
    '2011-2012': ('2011', 'G'),
    '2013-2014': ('2013', 'H'),
    '2015-2016': ('2015', 'I'),
    '2021-2023': ('2021', 'L'),
}
COMPONENTS = ['TST', 'DEMO', 'BMX', 'FASTQX', 'BIOPRO', 'RXQ_RX']
BASE = 'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{year}/DataFiles/{comp}_{suf}.xpt'

def fetch(cycle, comp):
    year, suf = CYCLES[cycle]
    d = RAW_DIR / cycle
    d.mkdir(parents=True, exist_ok=True)
    path = d / f'{comp}_{suf}.xpt'
    if not path.exists():
        url = BASE.format(year=year, comp=comp, suf=suf)
        print('downloading', url)
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=120) as r:
            path.write_bytes(r.read())
    return path

for cycle in CYCLES:
    for comp in COMPONENTS:
        p = fetch(cycle, comp)
        print(f'{cycle} {comp:7s} {p.stat().st_size/1e6:6.2f} MB')


2011-2012 TST       0.13 MB
2011-2012 DEMO      3.75 MB
2011-2012 BMX       1.95 MB
2011-2012 FASTQX    1.36 MB
2011-2012 BIOPRO    2.00 MB
2011-2012 RXQ_RX    2.99 MB
2013-2014 TST       0.47 MB
2013-2014 DEMO      3.83 MB
2013-2014 BMX       2.05 MB
2013-2014 FASTQX    1.44 MB
2013-2014 BIOPRO    2.13 MB
2013-2014 RXQ_RX    9.90 MB
2015-2016 TST       0.45 MB
2015-2016 DEMO      3.76 MB
2015-2016 BMX       1.99 MB
2015-2016 FASTQX    1.40 MB
2015-2016 BIOPRO    2.06 MB
2015-2016 RXQ_RX    9.63 MB
2021-2023 TST       2.38 MB
2021-2023 DEMO      2.58 MB
2021-2023 BMX       1.56 MB
2021-2023 FASTQX    1.33 MB
2021-2023 BIOPRO    2.43 MB
2021-2023 RXQ_RX    0.29 MB


## Merge and derive variables

* `session`: 0 = morning session. SAS transport files store the zero as
  a denormal float (≈5e-79), hence the `round()`.
* `race5`: NHANES `RIDRETH3` collapsed to the five groups the Army
  demographics report supports: Hispanic (any race), NH White, NH Black,
  NH Asian, NH other/multiracial.
* `trt`: any reported prescription whose name contains "TESTOSTERONE".
  The 2021–2023 prescription file omits drug names, so `trt` is missing
  for that cycle.
* `free_t`: Vermeulen calculated free testosterone from total T, SHBG and
  albumin (not available for 2011–2012, which lacks SHBG).
* `wt`: phlebotomy weight `WTPH2YR` for 2021–2023 (the TST panel ran on
  all phlebotomy participants); MEC weight `WTMEC2YR` for 2011–2016,
  where testosterone was measured on the full examined sample.


In [2]:
def load_cycle(cycle):
    year, suf = CYCLES[cycle]
    rd = lambda comp: pd.read_sas(RAW_DIR / cycle / f'{comp}_{suf}.xpt')
    tst, demo, bmx, fq, bio, rx = (rd(c) for c in COMPONENTS)
    sess_col = 'PHDSESNZ' if 'PHDSESNZ' in fq else 'PHDSESN'
    d = (demo.merge(tst, on='SEQN')
             .merge(bmx[['SEQN', 'BMXBMI']], on='SEQN', how='left')
             .merge(fq[['SEQN', sess_col, 'PHAFSTHR']], on='SEQN', how='left')
             .merge(bio[['SEQN', 'LBXSAL']], on='SEQN', how='left'))
    d['cycle'] = cycle
    d['session'] = d[sess_col].round().astype('Int64')
    d['wt'] = d['WTPH2YR'] if 'WTPH2YR' in d else d['WTMEC2YR']
    if 'LBXSHBG' not in d:
        d['LBXSHBG'] = np.nan
    if 'RXDDRUG' in rx:
        names = rx['RXDDRUG'].astype(str).str.upper()
        d['trt'] = d['SEQN'].isin(rx.loc[names.str.contains('TESTOSTERONE'), 'SEQN']).astype(float)
    else:
        d['trt'] = np.nan
    return d

frames = [load_cycle(c) for c in CYCLES]
allc = pd.concat(frames, ignore_index=True)
print(allc.groupby('cycle').size())


cycle
2011-2012    7821
2013-2014    8291
2015-2016    8021
2021-2023    8493
dtype: int64


In [3]:
men = allc[(allc['RIAGENDR'] == 1) & allc['RIDAGEYR'].between(18, 59) & allc['LBXTST'].notna()].copy()

men['age'] = men['RIDAGEYR'].astype(int)
men['bmi'] = men['BMXBMI']
men['total_t'] = men['LBXTST']
men['shbg'] = men['LBXSHBG']
men['albumin'] = men['LBXSAL']
men['free_t'] = vermeulen_free_t(men['total_t'], men['shbg'], men['albumin'])
men['race5'] = men['RIDRETH3'].round().map({1: 'Hispanic', 2: 'Hispanic', 3: 'NH White',
                                            4: 'NH Black', 6: 'NH Asian', 7: 'NH Other'})
men['age_grp'] = pd.cut(men['age'], [17, 25, 30, 35, 40, 59],
                        labels=['18-25', '26-30', '31-35', '36-40', '41-59']).astype(str)
men['morning'] = men['session'] == 0
men['fast_hr'] = men['PHAFSTHR']
men['strata'] = men['SDMVSTRA'].astype(int)
men['psu'] = men['SDMVPSU'].astype(int)

keep = ['SEQN', 'cycle', 'age', 'age_grp', 'race5', 'bmi', 'total_t', 'shbg', 'albumin',
        'free_t', 'morning', 'session', 'fast_hr', 'trt', 'wt', 'strata', 'psu']
men = men[keep].reset_index(drop=True)

flow = men.groupby('cycle').agg(men_18_59_with_T=('SEQN', 'size'),
                                morning=('morning', 'sum'),
                                morning_with_bmi=('bmi', lambda s: int((s.notna() & men.loc[s.index, 'morning']).sum())),
                                trt_users=('trt', 'sum'))
flow


,men_18_59_with_T,morning,morning_with_bmi,trt_users
cycle,,,,
2011-2012,1768,871,866,4.0
2013-2014,1863,881,876,7.0
2015-2016,1749,813,810,7.0
2021-2023,1443,784,780,0.0


In [4]:
out = DERIVED_DIR / 'nhanes_testosterone_men_18_59.parquet'
men.to_parquet(out, index=False)
print(f'wrote {out} ({len(men):,} rows)')
men.describe().T.round(2)


wrote /home/user/ai_assisted_us_health_data_analysis/data/derived/nhanes_testosterone_men_18_59.parquet (6,823 rows)


,count,mean,std,min,25%,50%,75%,max
SEQN,6823.0,90394.474425,25125.440327,62161.0,71611.5,82448.0,92186.5,142309.0
age,6823.0,37.935805,12.432143,18.0,27.0,38.0,49.0,59.0
bmi,6777.0,28.606552,6.57243,14.1,24.1,27.5,31.8,74.1
total_t,6823.0,440.484022,206.082062,2.7,305.97,412.0,539.0,5180.0
shbg,4754.0,36.036035,19.264579,5.49,22.9725,32.15,44.115,290.7
albumin,6786.0,4.412762,0.328044,2.0,4.2,4.4,4.6,5.5
free_t,4718.0,8.954188,5.528138,0.190344,6.412234,8.297888,10.60554,201.799405
session,6823.0,0.646783,0.70977,0.0,0.0,1.0,1.0,2.0
fast_hr,6821.0,7.49245,5.439333,0.0,2.0,9.0,12.0,39.0
trt,5380.0,0.003346,0.057751,0.0,0.0,0.0,0.0,1.0
